# Yeom MEG 3D-reaching — Colab GPU runner

Trains the Brain2Qwerty-style **`convtransformer`** decoder on the Yeom 2023 MEG
reaching dataset using a Colab **NVIDIA GPU**.

**Flow:** mount Drive → download dataset → extract your subject to Drive → load → train.

**Before running:**
1. *Runtime → Change runtime type → Hardware accelerator: **GPU*** (a T4 is plenty).
2. The code repo is **public**, so the *Get the code* cell clones it with no token.

The per-subject `.mat` you extract is written to Drive, so re-running in a fresh
session skips the download. The Yeom `.mat` are MATLAB **v7.3** files — `mat73`
(installed below) reads them.

In [ ]:
import torch
!nvidia-smi -L
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> GPU'

In [ ]:
# Colab already has torch(+CUDA), numpy, scipy, scikit-learn. Add the rest:
# - mat73: reads the v7.3 (HDF5) Yeom .mat files
# - mne:   only needed for the optional CSP baseline
# - remotezip: only for the optional small-but-slow download cell
!pip -q install mat73 mne remotezip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the code (public repo — no token)

In [ ]:
import sys, os, subprocess

CODE_SOURCE = 'git'     # 'git' (public repo) or 'drive'
GIT_URL     = 'https://github.com/atticus-429/meg-movement-decoder.git'
DRIVE_CODE_DIR = '/content/drive/MyDrive/hcp_motor_decoder'   # used only if CODE_SOURCE='drive'

if CODE_SOURCE == 'git':
    if not os.path.isdir('/content/code'):
        subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, '/content/code'], check=True)
    else:                                  # already cloned -> sync to latest main
        subprocess.run(['git', '-C', '/content/code', 'fetch', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', '/content/code', 'reset', '--hard', 'origin/main'], check=True)
    PKG = '/content/code'          # repo root IS the package
    print('code at', subprocess.run(['git', '-C', '/content/code', 'rev-parse', '--short', 'HEAD'],
                                     capture_output=True, text=True).stdout.strip())
else:
    PKG = DRIVE_CODE_DIR
assert os.path.isdir(PKG), f'code folder not found: {PKG}'
if PKG not in sys.path:
    sys.path.insert(0, PKG)
# drop any stale cached project modules so a fresh pull is actually re-imported
# (re-running this cell after a code change then suffices -- no kernel restart needed)
for _m in [k for k in list(sys.modules)
           if k in ('config', 'decode', 'evaluate', 'transfer') or k.startswith('loaders')]:
    del sys.modules[_m]
import config, decode, evaluate
from loaders.yeom import load_yeom
print('code OK at', PKG)

## 2. Configure dataset + download target

In [ ]:
# dataset variant: 'ica' (artifact-cleaned, recommended) or 'epoched'
DATA_VARIANT  = 'ica'
FIGSHARE_FILE = {'ica': '41898840', 'epoched': '41898714'}[DATA_VARIANT]
ZIP_URL = f'https://ndownloader.figshare.com/files/{FIGSHARE_FILE}'   # one ~9.3 GB zip

# extracted per-subject .mat live here on Drive (persists across sessions)
DRIVE_DATA_DIR = '/content/drive/MyDrive/yeom_meg/data'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# which subject/session to extract: a SUBSTRING of the .mat filename.
# Archive filenames are 'Sub_<n>_ses_<s>_ICA.mat' (ica) / 'Sub_<n>_ses_<s>.mat' (epoched),
# n = 1..9, s = 1..2. The download cell prints the full list.
SUBJECT_TOKEN = 'Sub_1_ses_1'

# keep the big 9.3 GB zip on local disk (fast, ephemeral) -> spares Drive quota.
# set True only if you want to extract many subjects across sessions without re-downloading.
ZIP_TO_DRIVE = False

# --- Tier-1 regularization preset for convtransformer (generalizable; NOT Yeom-tuned) ---
# TIER1=True adds: cosine LR + warmup, label smoothing, early stopping (held-out val,
# best weights restored), and light augmentation (sensor noise / time-jitter / channel-
# drop / mixup). All standard, low-data regularizers -> they transfer to other datasets.
# TIER1=False reproduces the original stock decoder. Output files are tagged '_tier1' so
# Tier-1 runs never overwrite a baseline CSV -> clean A/B against the prior numbers.
TIER1 = True
TIER1_KWARGS = dict(lr_schedule='cosine', warmup_frac=0.1, label_smoothing=0.1,
                    early_stopping=True, val_frac=0.2, patience=15,
                    noise_std=0.1, time_jitter=3, channel_drop=0.1, mixup_alpha=0.2)

def conv_kwargs(name):
    # Tier-1 kwargs apply ONLY to convtransformer (build_decoder rejects them for csp etc.)
    return TIER1_KWARGS if (TIER1 and name == 'convtransformer') else {}

def tier1_tag(name):
    return '_tier1' if (TIER1 and name == 'convtransformer') else ''

## 3. Download + extract your subject to Drive

Downloads the full ~9.3 GB archive to **fast local disk** and extracts only the
`.mat` matching `SUBJECT_TOKEN` (~0.5 GB) to Drive. Reliable, resumable (`wget -c`),
and idempotent — if your subject is already on Drive it skips everything.

In [ ]:
import glob, zipfile

def drive_mats():
    fs = glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True)
    return [f for f in fs if (not SUBJECT_TOKEN) or SUBJECT_TOKEN in os.path.basename(f)]

if drive_mats():
    print('already on Drive (skipping download):', [os.path.basename(f) for f in drive_mats()])
else:
    ZIP_PATH = (f'/content/drive/MyDrive/yeom_meg/yeom_{DATA_VARIANT}.zip'
                if ZIP_TO_DRIVE else f'/content/yeom_{DATA_VARIANT}.zip')
    os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
    if not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 9_000_000_000):
        print(f'downloading {DATA_VARIANT} archive (~9.3 GB) -> {ZIP_PATH} ...')
        rc = os.system(f'wget -c -O "{ZIP_PATH}" "{ZIP_URL}"')   # -c resumes if interrupted
        assert rc == 0 and os.path.getsize(ZIP_PATH) > 9_000_000_000, \
            'download incomplete — re-run this cell to resume'
    with zipfile.ZipFile(ZIP_PATH) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        print(f'{len(mats)} .mat in archive, e.g. {mats[:3]}')
        sel = [n for n in mats if SUBJECT_TOKEN in n] if SUBJECT_TOKEN else mats
        assert sel, f'no .mat matched SUBJECT_TOKEN={SUBJECT_TOKEN!r}; choose from the list above'
        for n in sel:
            print('extracting', n, '-> Drive'); z.extract(n, DRIVE_DATA_DIR)
    assert drive_mats(), 'extraction produced no .mat on Drive!'
    print('on Drive now:', [os.path.basename(f) for f in drive_mats()])

### (Optional) small download via remotezip

Pulls **only** your subject (~0.5 GB) straight to Drive without the full archive,
but is **~10-15 min** because figshare's download URLs expire every 10 s (each
chunk re-redirects). Use only if you can't spare ~10 GB of local disk. Off by default.

In [ ]:
RUN_REMOTEZIP = False
if RUN_REMOTEZIP and not glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True):
    from remotezip import RemoteZip
    assert SUBJECT_TOKEN, 'set SUBJECT_TOKEN'
    with RemoteZip(ZIP_URL) as z:
        sel = [n for n in z.namelist() if n.lower().endswith('.mat') and SUBJECT_TOKEN in n]
        assert sel, f'no .mat matched {SUBJECT_TOKEN!r}'
        for n in sel:
            print('downloading (slow)', n, '...'); z.extract(n, DRIVE_DATA_DIR)
    print('done')

## 4. Load the subject from Drive

In [ ]:
import numpy as np
X, y, sfreq, names = load_yeom(DRIVE_DATA_DIR, subject=SUBJECT_TOKEN or None,
                              sensor_type='all', resample=150, crop=(0.0, 1.5))
print('X', X.shape, '| classes', names, '| counts', np.bincount(y).tolist(), '| sfreq', sfreq)

## 5. Train the conv→transformer (GPU)

With a GPU, full 5-fold CV is feasible (`CV='kfold'`); use `'holdout'` for a quick
single-split pass. The decoder auto-uses CUDA. `TIER1` (set in §2) toggles the
regularization preset; evaluate with **5-fold** — a single holdout is high-variance
and can look degenerate even when CV is fine.

In [ ]:
from decode import build_decoder
from evaluate import bootstrap_error_ci, format_confusion
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

CV = 'kfold'   # 'kfold' (5-fold, GPU) or 'holdout' (single split, fastest)
dec = build_decoder('convtransformer', sfreq=sfreq, **conv_kwargs('convtransformer'))
print('TIER1', TIER1, '| convtransformer kwargs:', conv_kwargs('convtransformer'))

if CV == 'kfold':
    cvs = StratifiedKFold(5, shuffle=True, random_state=0)
    y_pred, y_eval = cross_val_predict(dec, X, y, cv=cvs, n_jobs=1), y
else:
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0)
    dec.fit(X[tr], y[tr]); y_pred, y_eval = dec.predict(X[te]), y[te]

err, lo, hi = bootstrap_error_ci(y_eval, y_pred)
chance = 100.0 / len(names)
print(f'convtransformer | accuracy {100*(1-err):.1f}%  '
      f'95% CI [{100*(1-hi):.1f}, {100*(1-lo):.1f}]  chance {chance:.1f}%')
cs, cm = format_confusion(y_eval, y_pred, names)
print(cs)

os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/result_convtransformer{tier1_tag("convtransformer")}.npz'
np.savez(out, y_true=y_eval, y_pred=y_pred, cm=cm, acc=1 - err, classes=names)
print('saved', out)

## 6. (Optional) CSP baseline for comparison

In [ ]:
dec = build_decoder('csp', sfreq=sfreq)
yp = cross_val_predict(dec, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print('csp baseline accuracy:', round(100 * np.mean(yp == y), 1), '%')

## 7. Multi-subject window-comparison suite (GPU)

For each subject-session, runs the chosen decoder under THREE time windows and
aggregates a per-session table + mean +/- spread to Drive:
- **peri**   - cue-locked [0, 1.5 s]            (the original setting)
- **pre**    - accel-gated [onset-0.5 s, onset]  (pre-movement; artifact-controlled)
- **during** - accel-gated [onset, onset+0.5 s]  (execution; equal-length control)

Extracts all 18 `.mat` to local `/content` (~9.3 GB). **WARNING:** the full
18-session x 3-window conv->transformer sweep is long (~1.5-2.5 h on a T4). Set
`SUBSET=['Sub_1']` first for a quick (~3-run) sanity check, comment out windows, or
use `SUITE_DECODER='csp'`.

In [ ]:
# extract ALL subject/session .mat to local disk (not Drive -> spares quota)
import glob, zipfile
ALL_DIR = '/content/yeom_all'
os.makedirs(ALL_DIR, exist_ok=True)
allmats = glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True)
if len(allmats) >= 18:
    print(f'{len(allmats)} .mat already extracted at {ALL_DIR}')
else:
    ZIP_PATH = f'/content/yeom_{DATA_VARIANT}.zip'
    if not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 9_000_000_000):
        print('downloading full archive (~9.3 GB) ...')
        rc = os.system(f'wget -c -O "{ZIP_PATH}" "{ZIP_URL}"')
        assert rc == 0 and os.path.getsize(ZIP_PATH) > 9_000_000_000, 'download incomplete; re-run'
    with zipfile.ZipFile(ZIP_PATH) as z:
        mats = [n for n in z.namelist() if n.lower().endswith('.mat')]
        print(f'extracting {len(mats)} .mat to {ALL_DIR} ...')
        for n in mats:
            z.extract(n, ALL_DIR)
    allmats = glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True)
print(f'{len(allmats)} subject-session files ready')

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SUITE_DECODER = 'convtransformer'   # or 'csp'
SENSOR  = 'all'                     # 'all' / 'mag' / 'grad'
SUBSET  = None                      # e.g. ['Sub_1'] for a quick check; None = all 18 sessions
WINDOWS = [                         # (name, align, crop); comment out rows to skip
    ('peri',   'cue',      (0.0, 1.5)),
    ('pre',    'movement', (-0.5, 0.0)),
    ('during', 'movement', (0.0, 0.5)),
]

files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
if SUBSET:
    files = [f for f in files if any(s in os.path.basename(f) for s in SUBSET)]
print('TIER1', TIER1, '| suite decoder:', SUITE_DECODER,
      '| kwargs:', conv_kwargs(SUITE_DECODER))

def label(path):
    m = re.search(r'Sub_\d+_ses_\d+', os.path.basename(path))
    return m.group(0) if m else os.path.basename(path)

cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for f in files:
    name = label(f); rec = {'subject': name}
    for wname, align, crop in WINDOWS:
        X, y, sfreq, names = load_yeom(f, sensor_type=SENSOR, resample=150, crop=crop, align=align)
        dec = build_decoder(SUITE_DECODER, sfreq=sfreq, **conv_kwargs(SUITE_DECODER))
        yp = cross_val_predict(dec, X, y, cv=cv, n_jobs=1)
        acc = 100 * np.mean(yp == y)
        rec[wname] = round(acc, 1); rec[wname + '_n'] = int(len(y))
        print(f'{name} [{wname}]: {acc:.1f}%  (n={len(y)})')
    rows.append(rec)

df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
for wname, _, _ in WINDOWS:
    if wname in df:
        print(f'{wname:7s}: mean {df[wname].mean():.1f}% +/- {df[wname].std():.1f}  '
              f'(chance 25%, n={int(df[wname].notna().sum())} sessions)')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/window_comparison_{SUITE_DECODER}{tier1_tag(SUITE_DECODER)}_{SENSOR}.csv'
df.to_csv(out, index=False); print('saved', out)

## 8. Leave-one-subject-out (cross-subject transfer)

Trains on 8 subjects (pooled) and tests on the held-out 9th -- the real "does it
transfer to a NEW brain" test. This is **zero-shot**: the held-out subject gets no
calibration; the conv->transformer's shared 1x1 spatial layer is the cross-subject
input mapping. (The per-subject input layer + *calibration* variant -- Brain2Qwerty/
Willett style -- is the natural follow-on if zero-shot is weak.)

Default window = `during` (strongest within-subject, best chance of detecting
transfer); set `LOSO_WINDOW` to `('pre','movement',(-0.5,0.0))` for the
project-relevant pre-movement transfer. ~20-40 min on a T4 (9 trains on ~1920 trials).

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from evaluate import bootstrap_error_ci
from loaders.yeom import load_yeom

LOSO_DECODER = 'convtransformer'                    # or 'csp'
SENSOR       = 'all'
LOSO_WINDOW  = ('during', 'movement', (0.0, 0.5))   # (name, align, crop)
wname, align, crop = LOSO_WINDOW

# load every session, pool the 2 sessions per subject
files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
subj, sfreq = {}, 150.0
for f in files:
    s = re.search(r'Sub_\d+', os.path.basename(f)).group(0)
    X, y, sfreq, names = load_yeom(f, sensor_type=SENSOR, resample=150, crop=crop, align=align)
    subj.setdefault(s, []).append((X, y))
subjects = sorted(subj, key=lambda s: int(s.split('_')[1]))
pooled = {s: (np.concatenate([x for x, _ in subj[s]]),
              np.concatenate([y for _, y in subj[s]])) for s in subjects}
print('subjects:', subjects, '| trials each:', {s: len(pooled[s][1]) for s in subjects})

rows = []
for held in subjects:
    Xtr = np.concatenate([pooled[s][0] for s in subjects if s != held])
    ytr = np.concatenate([pooled[s][1] for s in subjects if s != held])
    Xte, yte = pooled[held]
    dec = build_decoder(LOSO_DECODER, sfreq=sfreq, **conv_kwargs(LOSO_DECODER))
    dec.fit(Xtr, ytr)
    yp = dec.predict(Xte)
    err, lo, hi = bootstrap_error_ci(yte, yp)
    acc = 100 * np.mean(yp == yte)
    rows.append(dict(held_out=held, n_train=int(len(ytr)), n_test=int(len(yte)),
                     acc=round(acc, 1), ci_low=round(100*(1-hi), 1), ci_high=round(100*(1-lo), 1)))
    print(f'{held}: {acc:.1f}%  [{100*(1-hi):.1f}, {100*(1-lo):.1f}]  (train {len(ytr)}, test {len(yte)})')

df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
print(f'\nzero-shot LOSO [{wname}] {LOSO_DECODER}: mean {df.acc.mean():.1f}% +/- {df.acc.std():.1f}  '
      f'(chance 25%, n={len(df)} held-out subjects)')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/loso_{LOSO_DECODER}{tier1_tag(LOSO_DECODER)}_{wname}_{SENSOR}.csv'
df.to_csv(out, index=False); print('saved', out)

## 9. Imagery test -- Rathee MEG motor/cognitive imagery (separate 52 GB dataset)

Does the pipeline reach imagery-decoding levels on a REAL imagery dataset (vs
Yeom's overt execution)? Downloads the Rathee BIDS `.fif` archive to local
`/content`, extracts the chosen subjects, and runs the conv->transformer
within-subject on the imagery window (0.5-3.5 s) -- both 4-class (hand/feet/
word-gen/subtraction) and the pure-motor 2-class (hand vs feet).

Reference (the SOTA bar for this data): intra-session within-subject **binary
~96-99%** (filter-bank CSP); 4-class is lower; inter-session ~56-69%.

**WARNING:** 52 GB download -- needs Colab Pro + a big disk, ~1 h. Start with one
subject (`RATHEE_SUBJECTS = ['01']`). On first run, check the printed STIM event
codes (expect 4/8/16/32, ~50 each) before trusting the accuracy.

**Persistence:** the extracted subjects are written to **Google Drive** (~3 GB
each, so they survive a runtime disconnect), and the cell **skips the download
entirely if the requested subjects are already on Drive**. Only the big zip lives
on ephemeral `/content` (it's just the intermediate; re-downloaded only if you
later add a subject that isn't on Drive yet).

In [ ]:
!pip -q install mne-bids
import os, glob, zipfile
RATHEE_SUBJECTS = ['01']                       # start with 1; add '02','03',... later

# extracted subjects -> Google DRIVE (persists across runtime disconnects, ~3 GB each)
RATHEE_BIDS = '/content/drive/MyDrive/rathee_meg/bids'
os.makedirs(RATHEE_BIDS, exist_ok=True)
# the 52 GB zip -> ephemeral local disk (only the intermediate for extraction).
# Set RZIP to a Drive path instead if you have ~52 GB free and want to keep it.
RZIP = '/content/MEG_BIDS.zip'
RURL = 'https://ndownloader.figshare.com/files/26723708'   # MEG_BIDS.zip, 52.3 GB

def _on_drive(s):
    return bool(glob.glob(os.path.join(RATHEE_BIDS, '**', f'sub-{s}', '**', '*.fif'),
                          recursive=True))

missing = [s for s in RATHEE_SUBJECTS if not _on_drive(s)]
if not missing:
    print('all requested subjects already on Drive', RATHEE_SUBJECTS, '-> skip download/extract')
else:
    if not (os.path.exists(RZIP) and os.path.getsize(RZIP) > 52_000_000_000):
        print(f'downloading MEG_BIDS.zip (~52 GB) for missing subjects {missing} ...')
        rc = os.system(f'wget -c -O "{RZIP}" "{RURL}"')
        assert rc == 0 and os.path.getsize(RZIP) > 52_000_000_000, 'download incomplete; re-run'
    with zipfile.ZipFile(RZIP) as z:                # extract only the missing subjects -> Drive
        want = [n for n in z.namelist()
                if any(f'sub-{s}/' in n.lower() for s in missing)
                or n.lower().rstrip('/').endswith(('participants.tsv', 'dataset_description.json'))]
        print(f'extracting {len(want)} files for {missing} -> Drive ...')
        for n in want:
            z.extract(n, RATHEE_BIDS)

roots = glob.glob(os.path.join(RATHEE_BIDS, '**', 'sub-*'), recursive=True)
BIDS_ROOT = os.path.dirname(sorted(roots)[0]) if roots else RATHEE_BIDS
print('BIDS root (on Drive):', BIDS_ROOT, '| subjects:',
      sorted({os.path.basename(r) for r in roots}))

In [ ]:
import numpy as np
from decode import build_decoder
from evaluate import bootstrap_error_ci
from loaders.rathee import load_rathee
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SUBJECT, SESSION = RATHEE_SUBJECTS[0], '1'
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for label, classes in [('4-class', None), ('2-class hand/feet', ('hand', 'feet'))]:
    X, y, sfreq, names = load_rathee(BIDS_ROOT, SUBJECT, SESSION,
                                     sensor_type='all', classes=classes, resample=150)
    dec = build_decoder('convtransformer', sfreq=sfreq)
    yp = cross_val_predict(dec, X, y, cv=cv, n_jobs=1)
    err, lo, hi = bootstrap_error_ci(y, yp)
    print(f'>> {label}: {100*np.mean(yp==y):.1f}%  '
          f'[{100*(1-hi):.1f}, {100*(1-lo):.1f}]  chance {100/len(names):.0f}%')

## 10. Tier-2 — cross-subject pretrain → per-subject calibration (WITHIN-session)

Tier-1 didn't move the pre-movement number → it's data-limited, not optimization-
limited. This borrows data ACROSS subjects: pretrain on the other subjects, then
calibrate on a few of the target's own trials. We confirmed cross-SESSION decoding
collapses (genuine covariate shift, ~head reposition between days) — but real BCIs
recalibrate each session, so the deployment-relevant test is **within-session**.

For each held-out subject and session: hold out `TEST_PER_CLASS`/class as a FIXED
test set; calibrate on K/class from the rest; test within the SAME session.
- **zeroshot** (K=0): pretrained model, no calibration (cross-subject → ~chance)
- **full (A)**: fine-tune all weights on K trials
- **frozen (C)**: freeze the trunk, adapt input adapter + head on K trials
- **scratch**: a fresh model on the SAME K trials — the baseline that decides whether
  pretraining adds data-efficiency (full/frozen > scratch at small K) or not.

**Machinery check (on `during`): scratch at the largest K should be high (~70–85%)** —
that confirms within-session decoding works and the split is sound. The Tier-2 question
is whether full/frozen BEAT scratch at SMALL K. **Validate `during` first, then set
`TRANSFER_WINDOW` to `pre`** (the target). Pretrained models cached to Drive (window-
specific) for resume; `SUBSET` for a cheap pass. (`cross_session` protocol — calibrate
one day, use another — is available via `PROTOCOL` as the harder stretch goal.)

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from transfer import pretrain_then_calibrate, summarize

TRANSFER_WINDOW = ('during', 'movement', (0.0, 0.5))   # validate; then ('pre','movement',(-0.5,0.0))
PROTOCOL    = 'within_session'         # realistic BCI (recalibrate each session); 'cross_session' = stretch goal
SENSOR      = 'all'
TEST_PER_CLASS = 10                    # fixed held-out test/class (40 test); calibration pool = 20/class
K_PER_CLASS = [0, 5, 10, 20]           # calibration trials/class drawn from the pool
SUBSET      = ['Sub_1', 'Sub_2', 'Sub_3']   # cheap validation pass; set None for all 9
wname, align, crop = TRANSFER_WINDOW
SAVE_DIR    = f'/content/drive/MyDrive/yeom_meg/pretrain_ckpt_{wname}'   # window-specific cache (resume)

# load every session, grouped by subject, sessions kept SEPARATE
files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
subjects, sfreq = {}, 150.0
for f in files:
    msub = re.search(r'Sub_\d+', os.path.basename(f)).group(0)
    mses = re.search(r'ses_(\d+)', os.path.basename(f))
    ses = 'ses' + (mses.group(1) if mses else '1')
    if SUBSET and msub not in SUBSET:
        continue
    X, y, sfreq, names = load_yeom(f, sensor_type=SENSOR, resample=150, crop=crop, align=align)
    subjects.setdefault(msub, {})[ses] = (X, y)
subjects = {s: d for s, d in subjects.items() if 'ses1' in d and 'ses2' in d}
print('TIER1', TIER1, '| protocol', PROTOCOL, '| window', wname, '| subjects:', sorted(subjects))

def build_fn():
    return build_decoder('convtransformer', sfreq=sfreq, **conv_kwargs('convtransformer'))

rows = pretrain_then_calibrate(subjects, build_fn, K_per_class_list=K_PER_CLASS,
                               protocol=PROTOCOL, test_per_class=TEST_PER_CLASS,
                               finetune_lr=1e-4, finetune_epochs=30, save_dir=SAVE_DIR)

df = pd.DataFrame(rows); summ = summarize(rows)
print(f'\n--- {PROTOCOL} calibration curve [{wname}]: mean over held-out subjects (acc%), chance 25% ---')
print(f"{'K':>3}  {'zeroshot':>9} {'full(A)':>8} {'frozen(C)':>10} {'scratch':>8}")
for K in sorted({k for k, _ in summ}):
    def g(v): return f'{summ[(K, v)][0]:.1f}' if (K, v) in summ else '   -'
    print(f"{K:>3}  {g('zeroshot'):>9} {g('full'):>8} {g('frozen'):>10} {g('scratch'):>8}")
print('\nMACHINERY (during): scratch at the largest K should be high (~70-85%).'
      '\nKEY TEST: full/frozen vs scratch at SMALL K -- beats scratch => pretraining adds data-efficiency.')

os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/calibration_{PROTOCOL}_{wname}{tier1_tag("convtransformer")}_{SENSOR}.csv'
df.to_csv(out, index=False); print('saved', out)

## 11. Tier-2 cross-WINDOW transfer — lift `pre` using the strong `during` window

`pre` is data/signal-limited; its richest ally is the SAME trials' strong `during`
half (same direction labels, high SNR). Load the COMBINED **[-0.5,+0.5]s** movement-
onset window once and slice it 50/50 → early half = **PRE** (planning), late half =
**DURING** (execution); both halves are 306×75 so one model fits both.

Per subject/session: split trials into CAL/TEST; **pretrain the feature-extractor on
DURING of CAL trials**; **calibrate on PRE of CAL trials**; **test on PRE of TEST
trials**. `scratch` = PRE-only baseline; `zeroshot` = DURING-pretrained with no PRE
calibration. No leakage — TEST trials' slices (either half) never enter pretrain/
calibrate, and the PRE test data has no movement (no execution-artifact leakage).

**KEY TEST: does `full`/`frozen` beat `scratch` on held-out PRE?** Yes → the strong
window's directional features transfer to planning (lifting ~41%). ≈ scratch → the
planning signal is window-specific and during-pretraining can't conjure it. Set
`INCLUDE_OTHERS=True` to also fold in every other subject's DURING (bigger corpus,
slower). during/peri keep their own decoders — untouched.

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from transfer import cross_window_transfer, summarize

SENSOR         = 'all'
TEST_PER_CLASS = 10                    # fixed held-out PRE test/class
K_PER_CLASS    = [0, 5, 10, 20]        # PRE calibration trials/class
INCLUDE_OTHERS = False                 # True = also pretrain on other subjects' DURING (bigger, slower)
SUBSET         = ['Sub_1', 'Sub_2', 'Sub_3']   # cheap pass; None = all 9
SAVE_DIR       = '/content/drive/MyDrive/yeom_meg/xwin_ckpt' if INCLUDE_OTHERS else None

# load the COMBINED [-0.5,+0.5]s movement-onset window once (sliced 50/50 -> PRE | DURING)
files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
subjects, sfreq = {}, 150.0
for f in files:
    msub = re.search(r'Sub_\d+', os.path.basename(f)).group(0)
    if SUBSET and msub not in SUBSET:
        continue
    mses = re.search(r'ses_(\d+)', os.path.basename(f))
    ses = 'ses' + (mses.group(1) if mses else '1')
    X, y, sfreq, names = load_yeom(f, sensor_type=SENSOR, resample=150,
                                   crop=(-0.5, 0.5), align='movement')
    subjects.setdefault(msub, {})[ses] = (X, y)
print('TIER1', TIER1, '| include_others', INCLUDE_OTHERS, '| subjects:', sorted(subjects),
      '| X_full:', {s: subjects[s][sorted(subjects[s])[0]][0].shape for s in subjects})

def build_fn():
    return build_decoder('convtransformer', sfreq=sfreq, **conv_kwargs('convtransformer'))

rows = cross_window_transfer(subjects, build_fn, K_per_class_list=K_PER_CLASS,
                             test_per_class=TEST_PER_CLASS, include_others_strong=INCLUDE_OTHERS,
                             finetune_lr=1e-4, finetune_epochs=30, save_dir=SAVE_DIR)

df = pd.DataFrame(rows); summ = summarize(rows)
print('\n--- cross-window (DURING->PRE) calibration curve: mean over subjects (acc%), chance 25% ---')
print(f"{'K':>3}  {'zeroshot':>9} {'full(A)':>8} {'frozen(C)':>10} {'scratch':>8}")
for K in sorted({k for k, _ in summ}):
    def g(v): return f'{summ[(K, v)][0]:.1f}' if (K, v) in summ else '   -'
    print(f"{K:>3}  {g('zeroshot'):>9} {g('full'):>8} {g('frozen'):>10} {g('scratch'):>8}")
print('\nKEY TEST: full/frozen vs scratch on held-out PRE -- beats scratch => the strong DURING'
      ' window lifts PRE; ~equal => planning signal is window-specific.')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
tag = '_others' if INCLUDE_OTHERS else ''
out = f'/content/drive/MyDrive/yeom_meg/calibration_crosswindow{tag}{tier1_tag("convtransformer")}_{SENSOR}.csv'
df.to_csv(out, index=False); print('saved', out)

## 12. Low-frequency / late-window PRE test

Literature: pre-movement DIRECTION lives in SLOW (<~7 Hz, esp. 0.1–3 Hz SCP) **time-
domain** potentials and concentrates in the **last ~100–300 ms before onset** — band
*power* gives ~chance (Waldert 2008 MEG 4-dir **67%** from <3 Hz; Lew 2014 EEG 4-dir
**76%** from 0.1–1 Hz SCP). Our pipeline z-scores broadband 150 Hz data, possibly
under-using that band. This sweeps **band × window** on the within-session PRE task
(5-fold), for the literature-faithful **`tdlinear`** (low-freq time-domain → L2-logistic,
fast/CPU) and our **`convtransformer`**, vs the no-filter baseline (~41.5%).

The filter is applied to the **full epoch before windowing** (no short-window edge
artifacts); accelerometer stays raw for onset detection. Principled + transfers to OPM.
`tdlinear` runs the full grid (cheap); `convtransformer` runs a small grid (GPU-heavy) —
expand `CONV_*` once you see which band/window wins.

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SUBSET   = ['Sub_1', 'Sub_2', 'Sub_3']          # None = all 9
BANDS    = [None, 3.0, 7.0, (0.1, 3.0)]         # None = broadband baseline; scalar = low-pass; tuple = band-pass
WINDOWS  = [(-0.5, 0.0), (-0.3, 0.0), (-0.2, 0.0)]   # full pre vs late (signal concentrates near onset)
CONV_BANDS   = [None, 3.0]                      # convtransformer is GPU-heavy -> small grid
CONV_WINDOWS = [(-0.5, 0.0), (-0.3, 0.0)]
RESAMPLE = {'tdlinear': 50, 'convtransformer': 150}   # tdlinear is low-freq -> 50 Hz is plenty + fewer features

files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
if SUBSET:
    files = [f for f in files if any(s in os.path.basename(f) for s in SUBSET)]
print('sessions:', len(files), '| baseline to beat: pre broadband (-0.5..0) ~41.5%, chance 25%')

def bstr(b):
    return 'none' if b is None else (f'<{b:g}' if np.isscalar(b) else f'{b[0]:g}-{b[1]:g}')

cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for dec_name in ['tdlinear', 'convtransformer']:
    bands = BANDS if dec_name == 'tdlinear' else CONV_BANDS
    windows = WINDOWS if dec_name == 'tdlinear' else CONV_WINDOWS
    for band in bands:
        for win in windows:
            accs = []
            for f in files:
                X, y, sfreq, names = load_yeom(f, sensor_type='all', resample=RESAMPLE[dec_name],
                                               crop=win, align='movement', band=band)
                dec = build_decoder(dec_name, sfreq=sfreq, **conv_kwargs(dec_name))
                yp = cross_val_predict(dec, X, y, cv=cv, n_jobs=1)
                accs.append(100 * np.mean(yp == y))
            m, s = float(np.mean(accs)), float(np.std(accs))
            rows.append(dict(decoder=dec_name, band=bstr(band),
                             window=f'{win[0]:+.2f}..{win[1]:+.2f}', acc=round(m, 1),
                             sd=round(s, 1), n=len(accs)))
            print(f'{dec_name:14s} band {bstr(band):7s} win {win[0]:+.2f}..{win[1]:+.2f}: '
                  f'{m:.1f}% +/- {s:.1f}', flush=True)

df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
print('\nRead: does any low-freq band x late window beat broadband (-0.5..0)? '
      'tdlinear gains => the slow-band signal is real; convtransformer gains => our pipeline can use it.')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
out = f'/content/drive/MyDrive/yeom_meg/pre_lowfreq_sweep{tier1_tag("convtransformer")}.csv'
df.to_csv(out, index=False); print('saved', out)

## 13. Confound checks for the ~78% pre-movement direction signal

The §12 result (~78% with low-freq + late window) could be motor PLANNING — or
anticipatory GAZE toward the target, or late cue/decision processing. With RT ~394 ms
there is no cue-free pre-movement gap to separate them, so two decisive controls:

1. **EOG-alone** — decode direction from the eye channel only. ~chance ⇒ the signal is
   NOT ocular; ~78% ⇒ it's gaze.
2. **Pre-CUE leakage** — decode direction from MEG in the 0.5 s BEFORE the cue appears
   (the subject doesn't yet know the direction). This MUST be ~chance (25%); above
   chance ⇒ drift / block-structure leakage inflating everything.

Either control coming out high invalidates the ~78% as a neural pre-movement signal.

In [ ]:
import os, re, glob, numpy as np, pandas as pd
from decode import build_decoder
from loaders.yeom import load_yeom
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SUBSET = ['Sub_1', 'Sub_2', 'Sub_3']            # None = all 9
files = sorted(glob.glob(os.path.join(ALL_DIR, '**', '*.mat'), recursive=True))
if SUBSET:
    files = [f for f in files if any(s in os.path.basename(f) for s in SUBSET)]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def sweep(sensor, align, crop, band, resample=50):
    accs = []
    for f in files:
        X, y, sf, names = load_yeom(f, sensor_type=sensor, resample=resample,
                                    crop=crop, align=align, band=band)
        yp = cross_val_predict(build_decoder('tdlinear', sfreq=sf), X, y, cv=cv, n_jobs=1)
        accs.append(100 * np.mean(yp == y))
    return float(np.mean(accs)), float(np.std(accs))

rows = []
print('--- (1) GAZE control: decode direction from EOG ALONE (high => ocular, not neural) ---')
for band, win in [(3.0, (-0.2, 0.0)), ((0.1, 3.0), (-0.2, 0.0)), (3.0, (-0.5, 0.0))]:
    m, s = sweep('eog', 'movement', win, band)
    rows.append(dict(test='EOG-alone', band=str(band), window=f'{win[0]:+.2f}..{win[1]:+.2f}',
                     acc=round(m, 1), sd=round(s, 1)))
    print(f'  EOG-alone band {band} win {win}: {m:.1f} +/- {s:.1f}   (MEG was ~78%)', flush=True)

print('--- (2) LEAKAGE control: decode direction from MEG BEFORE the cue (must be ~chance 25%) ---')
for band in [3.0, (0.1, 3.0)]:
    m, s = sweep('all', 'cue', (-0.5, 0.0), band)   # cue-locked (-0.5..0) = the 0.5 s BEFORE the cue
    rows.append(dict(test='preCUE-MEG', band=str(band), window='-0.50..0.00 (pre-cue)',
                     acc=round(m, 1), sd=round(s, 1)))
    print(f'  preCUE-MEG band {band}: {m:.1f} +/- {s:.1f}   (should be ~25%)', flush=True)

df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
print('\nVERDICT: EOG-alone ~chance => NOT ocular gaze; preCUE-MEG ~chance => no leakage.'
      ' Either one high invalidates the ~78% as a neural pre-movement direction signal.')
os.makedirs('/content/drive/MyDrive/yeom_meg', exist_ok=True)
df.to_csv('/content/drive/MyDrive/yeom_meg/pre_confound_check.csv', index=False); print('saved')

## 14. Pseudo-online causal replay — how much does `tdlinear` lose going CAUSAL?

The §12 PRE numbers come from an OFFLINE pipeline that uses the future: zero-phase
`sosfiltfilt`, accelerometer-onset windowing (the accel fires *after* the window),
acausal `resample_poly`, and a grid max over band×window. None of that is reproducible
online. This replays the same `tdlinear` PRE decode with every operation forced CAUSAL
and reports the accuracy lost:

- **filter** `sosfiltfilt` → causal `sosfilt` (lag delay, worst in the slow band)
- **window** onset-anchored (perfect-onset upper bound) → cue-anchored (realistic; RT jitter)
- **resample** `resample_poly` → causal anti-alias low-pass + integer subsample
- **normalise** per-(channel×timepoint) → per-channel (no online temporal template)

Conditions: `offline` (=§12), `c:filter`/`c:scale`/`c:resamp` (attribute the gap one
toggle at a time), `causal-onset` (all causal, perfect onset), `causal-cue` (all causal,
realistic).

**Protocol = nested CV** (`PO_PROTOCOL='nested'`): each outer fold selects band×window by
inner CV on calibration trials only, then LOCKS it before decoding the held-out test —
an honest deployable number, not the selection-inflated grid max (`'grid'` gives that
ceiling). Reports per-condition accuracy + 95% CI, the causal & alignment penalties, the
causal-filter lag (ms), and which band×window is selected (stability).

CPU-fine (`tdlinear` is linear). The onset-anchored conditions need the accelerometer
channels in the `.mat` — the printed "onset detected N/M" confirms they're present.

In [ ]:
import sys, os, glob
# refresh project modules so a fresh pull / edit is re-imported (mirror the get-code cell)
for _m in [k for k in list(sys.modules)
           if k in ('pseudo_online', 'config', 'decode', 'evaluate') or k.startswith('loaders')]:
    del sys.modules[_m]
import pseudo_online as po
from loaders.yeom import load_full_epochs

PO_PROTOCOL  = 'nested'      # 'nested' = honest online estimate; 'grid' = diagnostic ceiling
PO_SUBJECTS  = None          # None = every .mat on Drive; or e.g. ['Sub_1_ses_1']
PO_TARGET_FS = 50.0          # tdlinear is low-freq -> 50 Hz is plenty
PO_KOUTER, PO_KINNER, PO_SEED = 5, 4, 0
PO_OUT = f'/content/drive/MyDrive/yeom_meg/pseudo_online_{PO_PROTOCOL}.csv'

files = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '**', '*.mat'), recursive=True))
if PO_SUBJECTS:
    files = [f for f in files if any(s in os.path.basename(f) for s in PO_SUBJECTS)]
assert files, f'no .mat under {DRIVE_DATA_DIR} -- run the download/extract cell first'
print(len(files), 'session file(s):', [os.path.basename(f) for f in files])

rows = []
for f in files:
    name = os.path.basename(f)
    X, y, sfreq, onsets, cue, names = load_full_epochs(f)
    if PO_PROTOCOL == 'nested':
        res = po.run_nested(X, y, sfreq, onsets, cue, names, target_fs=PO_TARGET_FS,
                            k_outer=PO_KOUTER, k_inner=PO_KINNER, seed=PO_SEED)
        for c in po.CONDITIONS:
            r = res.get(c['name'])
            if r:
                rows.append(dict(subject=name, cond=c['name'], acc=r['acc'],
                                 lo=r['lo'], hi=r['hi'], n=r['n'],
                                 picked=po._pick_str(r['picks'])))
    else:
        for r in po.run_grid(X, y, sfreq, onsets, cue, names,
                             target_fs=PO_TARGET_FS, folds=PO_KOUTER, seed=PO_SEED):
            rows.append(dict(subject=name, **r))

import pandas as pd
df = pd.DataFrame(rows)
print('\n' + df.to_string(index=False))
os.makedirs(os.path.dirname(PO_OUT), exist_ok=True)
df.to_csv(PO_OUT, index=False); print('saved', PO_OUT)